In [1]:
from pyspark.sql import SparkSession

# Initialize Interactive Spark Session with limited resources
spark = SparkSession.builder \
    .appName("JupyterInteractive") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.cores", "2") \
    .config("spark.cores.max", "2") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,io.delta:delta-spark_2.12:3.1.0") \
    .config("spark.jars.ivy", "/tmp/.ivy2") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✅ Spark Session Ready with 2 Cores!")


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /tmp/.ivy2/cache
The jars for the packages stored in: /tmp/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0c999272-2b83-4bf2-ba4f-078fe7ea3f5f;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 376ms :: artifacts dl 15ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 fr

✅ Spark Session Ready with 2 Cores!


In [2]:
# 1. Load the Delta table from MinIO
df = spark.read.format("delta").load("s3a://delta-lake/bronze/hotels")

# 2. Print total rows to verify ingestion
total_hotels = df.count()
print(f"Total Hotels in Bronze Table: {total_hotels}\n")

# 3. Preview the data using the DataFrame API
# (We intentionally leave out the embedding columns here because 
# printing 384-dimensional arrays makes the screen unreadable!)
#print("--- Data Preview ---")
#df.select("id", "name", "country_code", "geohash", "combined_address").show(5, truncate=False)

# ==========================================
# 4. Create a Temp View to use pure SQL
# ==========================================
df.createOrReplaceTempView("hotels_view")

print("--- SQL Aggregation Example ---")
# Let's see how many hotels you have per country


26/02/24 15:28:29 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/02/24 15:28:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 3:=================================================>       (43 + 2) / 50]

Total Hotels in Bronze Table: 3608

--- SQL Aggregation Example ---


In [10]:
spark.sql("""
    SELECT 
        * 
    FROM hotels_view 
    WHERE providerId = 'grnconnect'
    limit 2
""").show()

+--------+--------------------+--------------+----------+---------------+------------+--------+-------+--------+----------+--------+----------+----------+------------------+--------------------+-----------+------------+--------------+-----------+--------------+---------------------+--------------------------+-------------------------+--------------------------+----------------------------+----------------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+-------------------------+--------------------+
|      id|                name|relevanceScore|providerId|providerHotelId|providerName|language|   type|category|starRating|distance|attributes|imageCount|availableSuppliers|    original_message|geoCode_lat|geoCode_long|contact_phones|contact_fax|contact_emails|contact_address_line1|contact_address_postalCode|contact_address_city_name|contact_address_state_name|contact_address_country_code|contact_address_c

In [4]:
spark.sql("""
    SELECT 
        distinct providerName 
    FROM hotels_view 
    limit 10
""").show()

[Stage 16:=============================>                            (1 + 1) / 2]

+------------+
|providerName|
+------------+
|  bookingcom|
|  grnconnect|
|         ean|
+------------+



In [5]:
spark.sql("""
    SELECT 
        count(*) 
    FROM hotels_view
""").show()

+--------+
|count(1)|
+--------+
|    3608|
+--------+



In [6]:
# 1. Load the Delta table from MinIO
df = spark.read.format("delta").load("s3a://test-delta-lake/delta/bronze/hotel_pairs")

# 2. Print total rows to verify ingestion
total_hotel_pairs = df.count()
print(f"Total Hotel Pairs in Bronze Table: {total_hotel_pairs}\n")

df.createOrReplaceTempView("old_hotel_pairs")

spark.sql("""
    SELECT 
        count(*) 
    FROM old_hotel_pairs
""").show()

Total Hotel Pairs in Bronze Table: 150745

+--------+
|count(1)|
+--------+
|  150745|
+--------+



In [7]:
# 1. Load the Delta table from MinIO
df = spark.read.format("delta").load("s3a://test-delta-lake/delta/bronze/hotels")

# 2. Print total rows to verify ingestion
total_hotels = df.count()
print(f"Total old Hotels in Bronze Table: {total_hotels}\n")

df.createOrReplaceTempView("old_hotels")

spark.sql("""
    SELECT 
        count(*) 
    FROM old_hotels
""").show()

Total old Hotels in Bronze Table: 6168

+--------+
|count(1)|
+--------+
|    6168|
+--------+



In [8]:
df = spark.read.format("delta").load("s3a://delta-lake/bronze/hotel_pairs")
df.createOrReplaceTempView("pairs_view")

spark.sql("""
    SELECT 
        * 
    FROM pairs_view
    limit 5
""").show()

+--------+--------+-----------------+-----------------+-------------+--------------------+--------------------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+------------------+-----------------------------+--------------+-------------------------+----------------------+---------------------------------+----------------+---------------------------+------------------+-------------------+-----------------+-------------+-------------------+-----------------+-----------------+---------------+----------------------+---------------------------------+-------------------+-----------------------------+----------------+------------+--------------+----------+------+----------+----------+--------------------+------------+--------------------+--------------------+---------------------------+----------------------------+------------------------------+---------------------+--------------------+----------------+------------+--